# PresetFit: train the look-transfer network

This notebook trains PresetFit's neural network on a free Colab GPU and scores it on the TST2K test set.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*, then *Save*.
2. Optional but recommended: a free [Hugging Face](https://huggingface.co/join) account. Create a *read* token under *Settings → Access Tokens*, then in Colab click the key icon on the left (*Secrets*), add a secret named `HF_TOKEN` with the token as its value, and switch on *Notebook access*. Without it, the 170,000-file download can be slowed down by Hugging Face's limits for anonymous users.
3. *Runtime → Run all*. Google will ask once for permission to use your Drive.

**What gets saved to your Google Drive** (folder `PresetFit`): the prepared data (about 1–2 GB, made once), the training checkpoints and the test results. If Colab disconnects, just *Run all* again: every step continues where it stopped.

The datasets are for non-commercial research use only (see the TST100K licence).

## 1. Settings

In [ ]:
# Where the code comes from: "github" (a branch of the repository) or "zip" (presetfit.zip in your Drive folder).
CODE_SOURCE = "github"
GITHUB_REPO = "https://github.com/tanmoy79/presetfit.git"
GITHUB_BRANCH = "model"

DRIVE_FOLDER = "/content/drive/MyDrive/PresetFit"

# Training examples: pairs of photos edited with the same preset, PER_PRESET photos per preset.
# 12 -> about 262,000 examples from a 10.6 GB download (instead of 78 GB for the full dataset).
# Changing it later needs the data folder in Drive to be deleted first (data/split.json).
PER_PRESET = 12

# Step 4: short trial runs to pick the image network (each on a tenth of the data).
COMPARE_BACKBONES = ["resnet18", "convnext_tiny", "dinov2_small"]
COMPARE_EPOCHS = 2
COMPARE_FRACTION = 0.1

# Step 5: the full training run with the winner.
FULL_EPOCHS = 4
BATCH_SIZE = 32

## 2. Connect Drive, check the GPU, install PresetFit

In [ ]:
import os, subprocess, shutil, json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_FOLDER)
DRIVE.mkdir(parents=True, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU. Choose Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", gpu.stdout.strip())

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Hugging Face token found.")
except Exception:
    print("No HF_TOKEN secret: downloads will be anonymous (slower, but they work).")

In [ ]:
CODE = Path("/content/presetfit")
if not CODE.exists():
    if CODE_SOURCE == "github":
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(CODE)], check=True)
    else:
        shutil.unpack_archive(str(DRIVE / "presetfit.zip"), "/content")
subprocess.run(["pip", "install", "-q", "-e", f"{CODE}[train,bench]"], check=True)
print("PresetFit installed.")

## 3. Prepare the data (once, about 30–60 minutes)

Downloads the original photos and 12 edited photos per preset from TST100K (about 25 GB, one folder at a time; folders already in Drive are skipped), shrinks every image to 256 × 256, downloads the TST2K test set, removes training photos that also appear in the test set, and saves the result to your Drive. Next time it is copied back from Drive in a few minutes.

In [ ]:
DATA = Path("/content/data")
saved = DRIVE / "data"
(saved / "packs").mkdir(parents=True, exist_ok=True)
DATA.mkdir(exist_ok=True)

if not (saved / "split.json").exists():
    # Packs are written straight to Drive, one folder at a time, so a disconnect loses at most one folder.
    if not (DATA / "packs").exists():
        os.symlink(saved / "packs", DATA / "packs")
    !python -m presetfit.learning.prepare --out /content/data --per-preset {PER_PRESET}
    shutil.make_archive(str(saved / "tst2k_512"), "tar", str(DATA), "tst2k_512")
    shutil.copy(DATA / "split.json", saved / "split.json")  # written last: marks the data complete
else:
    shutil.copy(saved / "split.json", DATA / "split.json")
    if not (DATA / "tst2k_512").exists():
        shutil.unpack_archive(str(saved / "tst2k_512.tar"), str(DATA))

print("Copying packs to the local disk for fast reading...")
shutil.copytree(saved / "packs", "/content/packs", dirs_exist_ok=True)
counts = json.loads((DATA / "split.json").read_text())["counts"]
print(json.dumps(counts, indent=2))

## 4. Compare the three image networks (short runs)

Each network gets the same short training run on a tenth of the data. The one with the lowest validation colour error is used for the full run. Lower is better; *unchanged photo* is the score for doing nothing.

In [ ]:
RUNS = DRIVE / "runs"
for backbone in COMPARE_BACKBONES:
    out = RUNS / f"compare_{backbone}"
    print(f"=== {backbone} ===")
    !python -m presetfit.learning.train --packs /content/packs --split /content/data/split.json --out "{out}" --backbone {backbone} --epochs {COMPARE_EPOCHS} --train-fraction {COMPARE_FRACTION} --batch-size {BATCH_SIZE}

In [ ]:
import csv
scores = {}
for backbone in COMPARE_BACKBONES:
    log = RUNS / f"compare_{backbone}" / "log.csv"
    rows = list(csv.DictReader(log.open()))
    scores[backbone] = min(float(r["val_de76"]) for r in rows)
    print(f"{backbone:15} validation colour error {scores[backbone]:.2f}   "
          f"(unchanged photo {float(rows[-1]['val_unchanged_de76']):.2f}, {rows[-1]['minutes']} min)")
BEST = min(scores, key=scores.get)
print("\nBest network:", BEST)

## 5. Full training with the best network

Saves progress to Drive every 10 minutes. If Colab disconnects, *Run all* again and it continues.

In [ ]:
MAIN = RUNS / f"main_{BEST}"
!python -m presetfit.learning.train --packs /content/packs --split /content/data/split.json --out "{MAIN}" --backbone {BEST} --epochs {FULL_EPOCHS} --batch-size {BATCH_SIZE}

## 6. Score on TST2K (2,000 test photos, 512 px)

The same test as PresetFit's first benchmark. Lower *dE2000* is better.

| method | dE2000 |
|---|---|
| do nothing | 9.45 |
| Reinhard colour transfer | 10.23 |
| PresetFit today | 10.93 |
| ICTone (12-billion-parameter model, paper) | 5.78 |
| best possible with PresetFit's 8 sliders | 3.90 |

`learned` makes the full-size photo with a colour lookup table; `learned_sliders` with the 8 sliders (so the edits list is exact).

In [ ]:
results = DRIVE / "results"
results.mkdir(exist_ok=True)
!python -m presetfit.evaluation.benchmark /content/data/tst2k_512 --methods identity,learned,learned_sliders --checkpoint "{MAIN}/best.pt" --device cuda --out "{results}/tst2k_{BEST}.csv"

When it finishes, the score table above is all PresetFit needs. Share it (or the `results` folder) and we decide the next step together.